# SimPro-ValueNetwork — Fine-Tuning

Load pretrained weights and fine-tune all model parameters on complete molecular strings and 17 property labels. Tokenizer, mapping, and scaler stay fixed.

Every run uses a new output directory and reads initialization weights without modifying them. Run [training validation](colab_comprehensive_validation.ipynb) first, then use this example with real data and the production network. For label generation, see [prepare_finetuning_dataset.ipynb](prepare_finetuning_dataset.ipynb).

Optional learning example; pretrained inference does not require this notebook.

Extract the project to `/content/SimPro-ValueNetwork`, or set `PROJECT_DIR` to your project root. Google Drive is optional: enable `USE_GOOGLE_DRIVE` and select your mounted project path. Checkpoints may be stored at any readable location.

Saved outputs are illustrative and use generalized paths. Rerun cells for results in your environment.


The project is SimPro-ValueNetwork; the distribution/import name is simpro_vn, with simpro-vn and simpro-vn-train console entrypoints. Use normal installation in a fresh runtime. Use [Comprehensive Validation](colab_comprehensive_validation.ipynb) for all functional checks in one run.



In [ ]:
USE_GOOGLE_DRIVE = False  # Enable only if your files are stored in Google Drive.
if USE_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')

from pathlib import Path
from datetime import datetime
from uuid import uuid4
import json
import subprocess
import sys

PROJECT_DIR = Path('/content/SimPro-ValueNetwork')
assert (PROJECT_DIR / 'pyproject.toml').is_file(), PROJECT_DIR
RUN_ID = datetime.now().strftime('%Y%m%d_%H%M%S') + '_' + uuid4().hex[:8]

## Installation

Select a Colab GPU runtime for GPU execution and confirm CUDA-enabled PyTorch. Restart the runtime after updating an imported package.


In [ ]:
%pip install "{PROJECT_DIR}[chemistry]"
import torch
print('PyTorch:', torch.__version__, '| CUDA:', torch.cuda.is_available())

## Select Data and Model

CSV must contain representation strings, 17 unstandardized property labels, and Train/Val assignments. Test rows are excluded from training and model selection. This example executes the first of 10 planned epochs and retains the schedule for resuming. The default is the included 300-row smoke CSV (100 Train, 100 Val, 100 Test). Change DATASET when fine-tuning on your own data.


In [ ]:
from simpro_vn.training import TrainingConfig, run_training
import hashlib

DATASET = PROJECT_DIR / 'examples/data/value_network_smoke.csv'
SOURCE_CHECKPOINT = PROJECT_DIR / 'weights/smiles/value_network_best_smiles_bpe_min_freq_5000_d6.pt'
ARTIFACT_DIR = Path('/content/training_runs') / ('finetune_' + RUN_ID)
ARTIFACT_DIR.mkdir(parents=True, exist_ok=False)
EXECUTION_MODE = 'cli'  # Choose 'cli' or 'api'.
MAX_EPOCHS_THIS_RUN = 1  # Set None to complete all planned epochs.

config = TrainingConfig(
    dataset_path=str(DATASET), output_dir=str(ARTIFACT_DIR / 'run'),
    checkpoint_path=str(SOURCE_CHECKPOINT), mode='finetune',
    representation='smiles', min_frequency=5000, depth=6,
    epochs=10, batch_size=2, accumulation_steps=1,
    learning_rate=2e-5, min_learning_rate=1e-6,
    schedule='discounted', gamma=0.95, precision='fp32', device='auto',
)
assert DATASET.is_file(), DATASET
assert SOURCE_CHECKPOINT.is_file(), SOURCE_CHECKPOINT

def file_sha256(path):
    result = hashlib.sha256()
    with Path(path).open('rb') as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b''):
            result.update(block)
    return result.hexdigest()

source_hash = file_sha256(SOURCE_CHECKPOINT)
print('Output:', ARTIFACT_DIR)

## Run Fine-Tuning

The default demonstrates the CLI. Set `EXECUTION_MODE='api'` to call the same training implementation directly from the cell. Incompatible rows stop preflight by default, with rejection details under data/. Use a new run after correcting inputs.


In [ ]:
from dataclasses import asdict

if EXECUTION_MODE == 'cli':
    config_file = ARTIFACT_DIR / 'config.json'
    config_file.write_text(json.dumps(asdict(config), indent=2), encoding='utf-8')
    command = ['simpro-vn-train', '--config', str(config_file)]
    if MAX_EPOCHS_THIS_RUN is not None:
        command.extend(['--max-epochs', str(MAX_EPOCHS_THIS_RUN)])
    print('Command:', command)
    subprocess.run(command, cwd=PROJECT_DIR, check=True)
    result = json.loads((Path(config.output_dir) / 'result.json').read_text())
elif EXECUTION_MODE == 'api':
    result = run_training(config, max_epochs=MAX_EPOCHS_THIS_RUN)
else:
    raise ValueError('EXECUTION_MODE must be cli or api')

assert file_sha256(SOURCE_CHECKPOINT) == source_hash
print('Initialization weights unchanged')
print(json.dumps(result, indent=2))
active_result = result


## Load New Weights

Best and last checkpoints are saved in the new directory. Load best for prediction below; update SAMPLE when changing representation. History losses use standardized units and temporal weighting, not original-unit RMSE.


In [ ]:
from simpro_vn import ValueNetworkPredictor

phase_dir = Path(result['directory'])
predictor = ValueNetworkPredictor.from_config(phase_dir / 'inference_config.json')
SAMPLE = 'CCO'
print(predictor.predict(SAMPLE))
print('best:', result['best_checkpoint'])
print('last:', result['last_checkpoint'])
print(json.loads((phase_dir / 'history.json').read_text()))

## Optional: Resume

Set `RESUME=True` to continue. Keep the same data and total planned epochs. Restore optimizer, scheduler, and random states from last into another new directory. The original run is retained. Once all epochs are complete, start a new fine-tuning schedule from best instead.


In [ ]:
from dataclasses import replace

RESUME = False
if RESUME:
    resumed_config = replace(
        config, checkpoint_path=None, resume_from=result['last_checkpoint'],
        output_dir=str(ARTIFACT_DIR / ('resumed_' + uuid4().hex[:8])),
    )
    resumed = run_training(resumed_config, max_epochs=1)
    assert file_sha256(SOURCE_CHECKPOINT) == source_hash
    active_result = resumed
    print(json.dumps(resumed, indent=2))

## Optional: Export Results

Files under `/content/` disappear with the runtime. Set EXPORT_ROOT to your persistent storage location and enable EXPORT_RESULTS to copy the entire run, including best and the resumable last checkpoint. An existing destination is rejected. After resuming, the latest training output is selected automatically.


In [ ]:
import shutil

EXPORT_RESULTS = False
EXPORT_ROOT = None  # Set a persistent directory, e.g. a mounted Drive folder.
SAVE_RUN = Path(active_result['directory']).parent
if EXPORT_RESULTS:
    if EXPORT_ROOT is None:
        raise ValueError('Set EXPORT_ROOT before exporting results')
    destination = Path(EXPORT_ROOT).expanduser() / RUN_ID
    shutil.copytree(SAVE_RUN, destination)
    print('Saved:', destination)
